# ASI06 Memory & Context Poisoning — Upload Artifacts & Run Evaluation

**OWASP Category**: ASI06 — Memory & Context Poisoning | **Risk Severity**: Critical

**Mapped LLM Categories**: LLM01, LLM08 (Vector/Embedding Weaknesses), LLM09 (Misinformation)

**ASI06 tests for**:
- Memory store injection (poisoned user preferences, fabricated history, safety policy overrides)
- RAG context poisoning (injected documents with fabricated policies, false citations, unauthorized grants)
- Long-context manipulation (false prior-agreement injection across multi-turn sessions)

In [ ]:
%pip install okareo python-dotenv --quiet

In [ ]:
import sys
from pathlib import Path
_nb = globals().get("__vsc_ipynb_file__", ".")
NOTEBOOK_DIR = Path(_nb).resolve().parent
CATEGORY_DIR = NOTEBOOK_DIR.parent
if str(CATEGORY_DIR.parent.parent) not in sys.path:
    sys.path.insert(0, str(CATEGORY_DIR.parent.parent))
from okareo.checks import ModelBasedCheck, CheckOutputType
from okareo.model_under_test import Driver
from owasp.common import init_okareo, parse_artifact, build_target, SINGLE_TURN_DRIVER_TEMPLATE
okareo, OKAREO_API_KEY = init_okareo()
print(f"✓ Okareo SDK initialized (key: ...{OKAREO_API_KEY[-5:]})")

In [ ]:
from owasp.common import parse_artifact

scenarios_dir = CATEGORY_DIR / "scenarios"
registered_scenarios = {}
for p in sorted(scenarios_dir.glob("*.jsonl")):
    name = f"ASI06-{p.stem}"
    sc = okareo.upload_scenario_set(scenario_name=name, file_path=str(p))
    registered_scenarios[name] = sc
    print(f"  ✓ {name}")

registered_checks = {}
for p in sorted((CATEGORY_DIR / "checks").glob("*.md")):
    data = parse_artifact(p)
    r = okareo.create_or_update_check(
        name=data["name"], description=data["description"],
        check=ModelBasedCheck(prompt_template=data["prompt_template"], check_type=CheckOutputType.PASS_FAIL),
    )
    registered_checks[data["name"]] = r.id
    print(f"  ✓ {data['name']}")

registered_drivers = {}
for p in sorted((CATEGORY_DIR / "drivers").glob("*.md")):
    data = parse_artifact(p, default_temperature=0.6)
    result = okareo.create_or_update_driver(driver=Driver(
        name=data["name"], prompt_template=data["prompt_template"], temperature=data["temperature"]
    ))
    registered_drivers[data["name"]] = result
    print(f"  ✓ {data['name']}")

In [ ]:
target = build_target(CATEGORY_DIR)
CTX_CHECK = "ASI06-context-poisoning-detector"
MAX_TURNS = 10

SINGLE_TURN_SCENARIOS = ["ASI06-memory-store-injection", "ASI06-rag-context-poisoning"]
MULTI_TURN_MAP = {"ASI06-long-context-manipulation": "ASI06-context-fabricator"}

all_results = {}
st_driver = Driver(name="pass-through", prompt_template=SINGLE_TURN_DRIVER_TEMPLATE, temperature=0)

for sname in SINGLE_TURN_SCENARIOS:
    if sname not in registered_scenarios: continue
    try:
        run = okareo.run_simulation(
            target=target, driver=st_driver,
            name=f"ASI06 Single-Turn — {sname}",
            api_key=OKAREO_API_KEY, first_turn="target",
            scenario=registered_scenarios[sname], max_turns=1, checks=[CTX_CHECK],
        )
        all_results[sname] = run
        print(f"  ✓ {sname}: {getattr(run, 'app_link', run.id)}")
    except Exception as e:
        all_results[sname] = None; print(f"  ✗ {sname}: {e}")

for sname, dname in MULTI_TURN_MAP.items():
    if sname not in registered_scenarios or dname not in registered_drivers: continue
    drv = registered_drivers[dname]
    try:
        run = okareo.run_simulation(
            target=target,
            driver=Driver(temperature=getattr(drv,"temperature",0.6), name=dname, prompt_template=drv.prompt_template),
            name=f"ASI06 Simulation — {sname}",
            api_key=OKAREO_API_KEY, first_turn="target",
            scenario=registered_scenarios[sname], max_turns=MAX_TURNS, checks=[CTX_CHECK],
        )
        key = f"{sname} (multi-turn)"
        all_results[key] = run
        print(f"  ✓ {key}: {getattr(run, 'app_link', run.id)}")
    except Exception as e:
        all_results[f"{sname} (multi-turn)"] = None; print(f"  ✗ {e}")

errors = sum(1 for r in all_results.values() if r is None)
print(f"\nASI06 Complete — Total: {len(all_results)} | Errors: {errors}")